# ML-04 — Search Intelligence Data Contract

**Author:** Mehak Zahra  
**Lane:** Refresh / Content Opportunity Scoring  
**Decision:** Which measurable content pages should an editor review first?

This notebook queries the gated FlyRank warehouse with DuckDB. It never downloads the full daily table and never stores a token, client name, domain, URL, or raw query.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn pandas

import os
from getpass import getpass
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, KeyError, TypeError):
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = getpass('Hugging Face READ token (input is hidden): ')

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
print('Connected. March 2026 is the feature month; April 2026 is the outcome month.')

## 1. Contract in five plain-word answers

1. **One row:** One pseudonymized content page belonging to one pseudonymized client, after daily rows are aggregated.
2. **Tables:** `fact_content_daily_performance` provides daily search/analytics measurements. I use its March and April 2026 partitions only; no query-level or private-content table is needed.
3. **Time windows:** March 1–31, 2026 is the feature window. April 1–30, 2026 is the later outcome window. The decision moment is the end of March, so every retained feature is already knowable.
4. **What I predict/rank:** `declined_next_month = 1` when a page's average daily GSC impressions in April are more than 20% below its March average. Model probabilities rank pages for human review; they do not predict Google's algorithm or prove that refreshing causes recovery.
5. **Deliberate exclusion:** April measurements and `declined_next_month` are label-only. Pseudonymous IDs are context for joins and grouped splitting, never model features. I also exclude domains, URLs, raw queries, client names, and any product action flag.

Availability rule: the modeling slice requires `ga4_data_available IS TRUE`. This prevents pre-GA4 zero-filled rows from being interpreted as genuine zero engagement.

## 2. Exactly three verification queries

These three small queries inspect only the mid-panel March 2026 partition.

### Query 1 — grain

The documented daily grain is `report_date × client_hash_id × content_hash_id`. An empty output means no duplicate grain keys were found.

In [ ]:
grain_sql = f'''
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_grain
FROM {MARCH}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
'''
grain_violations = con.sql(grain_sql).df()
display(grain_violations)
assert grain_violations.empty, 'Daily grain has duplicates'

### Query 2 — slice row count and date span

This proves how many page-day rows and distinct pages are present and confirms the March window actually scanned.

In [ ]:
count_span_sql = f'''
SELECT
    COUNT(*) AS page_day_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_pages,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MARCH}
'''
count_span = con.sql(count_span_sql).df()
display(count_span)
assert str(count_span.loc[0, 'first_date'])[:10] == '2026-03-01'
assert str(count_span.loc[0, 'last_date'])[:10] == '2026-03-31'

### Query 3 — GA4 availability using `IS TRUE`

The first number is the full March slice; the second is the exact number surviving the availability filter. The percentage makes the cost of the filter visible.

In [ ]:
availability_sql = f'''
SELECT
    COUNT(*) AS all_page_day_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS available_page_day_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 2) AS available_pct
FROM {MARCH}
'''
availability = con.sql(availability_sql).df()
display(availability)
assert availability.loc[0, 'available_page_day_rows'] <= availability.loc[0, 'all_page_day_rows']

## 3. Five-feature frame and the deliberate leakage trap

The feature query aggregates March page-days, applies `ga4_data_available IS TRUE`, and joins an April outcome. Pages need at least 14 available days in each month and positive March impressions so unstable one-day histories do not dominate.

Exactly five honest features are retained:

| Feature | Available when? |
|---|---|
| `log_march_impressions` | Knowable at the decision moment because it uses only March GSC impressions observed before April. |
| `march_ctr_pct` | Knowable because March clicks and impressions have both landed before the end-of-March decision. |
| `march_avg_position` | Knowable because it averages only March GSC position observations; zero/no-data positions are converted to missing before averaging. |
| `march_active_days` | Knowable because it counts March days with at least one impression. |
| `march_position_sd` | Knowable because it measures variation only across March GSC position observations. |

`april_impressions` and `declined_next_month` belong to the label bucket and are never retained in the honest feature matrix.

In [ ]:
feature_sql = f'''
WITH march_page AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) AS march_days,
           SUM(gsc_impressions) AS march_impressions,
           SUM(gsc_clicks) AS march_clicks,
           AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS march_active_days,
           STDDEV_POP(NULLIF(gsc_avg_position, 0)) AS march_position_sd
    FROM {MARCH}
    WHERE ga4_data_available IS TRUE
    GROUP BY 1, 2
),
april_page AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) AS april_days,
           SUM(gsc_impressions) AS april_impressions
    FROM {APRIL}
    GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id,
       LN(1 + m.march_impressions) AS log_march_impressions,
       100.0 * m.march_clicks / NULLIF(m.march_impressions, 0) AS march_ctr_pct,
       m.march_avg_position,
       m.march_active_days,
       m.march_position_sd,
       a.april_impressions,
       CAST((a.april_impressions * 1.0 / a.april_days) <
            0.8 * (m.march_impressions * 1.0 / m.march_days) AS INTEGER) AS declined_next_month
FROM march_page m
JOIN april_page a USING (client_hash_id, content_hash_id)
WHERE m.march_days >= 14 AND a.april_days >= 14 AND m.march_impressions > 0
'''
feature_frame = con.sql(feature_sql).df()
FEATURES = [
    'log_march_impressions', 'march_ctr_pct', 'march_avg_position',
    'march_active_days', 'march_position_sd'
]
print(f'{len(feature_frame):,} page-level rows; label rate = {feature_frame.declined_next_month.mean():.3f}')
display(feature_frame[FEATURES + ['declined_next_month']].head())
assert len(FEATURES) == 5
assert feature_frame[FEATURES].notna().all(axis=1).any()

### Leakage experiment: make the score look suspiciously perfect, then delete the leak

I intentionally add `leaked_label_copy`, an exact copy of the outcome. This column is unknowable at the end-of-March decision moment. Its only purpose is to demonstrate why label-derived fields must be blocked. Both quick models use the same client-group split and random seed. The leaky AUC should jump to 1.0 or nearly 1.0; the honest AUC is the number I keep.

In [ ]:
model_frame = feature_frame.dropna(subset=FEATURES + ['declined_next_month']).copy()
X_honest = model_frame[FEATURES]
y = model_frame['declined_next_month']
groups = model_frame['client_hash_id']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X_honest, y, groups))

def quick_auc(columns):
    model = RandomForestClassifier(
        n_estimators=120, max_depth=6, min_samples_leaf=10,
        random_state=42, n_jobs=-1
    )
    model.fit(model_frame.iloc[train_idx][columns], y.iloc[train_idx])
    probability = model.predict_proba(model_frame.iloc[test_idx][columns])[:, 1]
    return roc_auc_score(y.iloc[test_idx], probability)

honest_auc = quick_auc(FEATURES)
model_frame['leaked_label_copy'] = model_frame['declined_next_month']  # deliberate trap
leaky_auc = quick_auc(FEATURES + ['leaked_label_copy'])

print(f'Honest grouped-holdout ROC AUC: {honest_auc:.3f}')
print(f'Leaky grouped-holdout ROC AUC:  {leaky_auc:.3f}')
assert leaky_auc > honest_auc

# Delete the trap. Only the five pre-decision features remain eligible.
model_frame.drop(columns='leaked_label_copy', inplace=True)
assert 'leaked_label_copy' not in model_frame.columns
KEPT_SCORE = honest_auc
print(f'Kept result: honest ROC AUC = {KEPT_SCORE:.3f}')

## 4. Named limitation

**Survivorship and availability limitation:** requiring `ga4_data_available IS TRUE`, at least 14 observed days in both months, and presence in both March and April removes clients/pages with shorter or incomplete histories. The resulting score therefore describes measurable continuing pages; it may not generalize to newly registered, GA4-unavailable, or discontinued pages. In addition, an observed April decline is a directional review proxy—not proof that a refresh would improve performance.

In [ ]:
public_columns = set(feature_frame.columns)
for forbidden in {'client_name', 'domain', 'url', 'query', 'title'}:
    assert forbidden not in public_columns
assert set(FEATURES).isdisjoint({'april_impressions', 'declined_next_month'})
print('Final checks passed: five honest features, label fields excluded, public-safe output.')

## 5. Self-check

- [x] Five plain-word contract answers are present.
- [x] Exactly three verification queries are shown with visible outputs after Run all.
- [x] Availability uses `ga4_data_available IS TRUE`.
- [x] The page-level frame contains exactly five pre-decision features, each with an “available when?” explanation.
- [x] One label-derived feature is deliberately added, its inflated score is shown, and it is deleted.
- [x] One named limitation is stated.
- [x] No client names, domains, URLs, private queries, or credentials are stored.
- [ ] Run all in Colab with the `HF_TOKEN` Secret, save the visible outputs, and commit this executed notebook.